# TPWRS Relevance to the 2025 Research Agenda: Evolution 2023-2026

Every TPWRS abstract in the benchmark runs is scored against all 48 RA2025 research questions by
several LLMs. This notebook reduces those scores to one number per abstract-question pair — the mean
across models — classifies that mean as **Low**, **Moderate** or **High** relevance, and tracks how
the mix moves across publication years.

Design choices:

- **RA questions only.** The PRP (programme-affinity) stage is not analysed here; programme-level
  views are built by grouping the RA questions instead.
- **Mean across models, `mistral-small-2603` excluded.** Its 2025 run is incomplete and its score
  distribution is the least discriminating of the panel, so it is dropped from the ensemble.
- **Levels follow `data/prompts/affinity_scoring_scale.yaml`:** Low `<= 40`, Moderate `40-80`,
  High `> 80`. ("Moderate" is the scale's name for the middle band.)
- **Programmes come from `data/processed/ra_grouping_ra2025.csv`** (`Grouping` x `RA2025`). An
  abstract's level for a programme is the highest score it reaches on any question in that
  programme's group, so levels are assigned per programme and are not mutually exclusive.
- **Shares, not counts.** Each year contains a different number of papers, so every trend is
  expressed as a percentage of that year's papers.

## 1. Configuration

New benchmark years are picked up automatically as long as their folder matches
`RESULT_FOLDER_PATTERN`. Plots use matplotlib defaults throughout — no custom style sheet, colour
palette or `rcParams` overrides.

In [ ]:
import re
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
BENCHMARK_ROOT = REPO_ROOT / "data" / "results" / "affinity_benchmark"
GROUPING_PATH = REPO_ROOT / "data" / "processed" / "ra_grouping_ra2025.csv"
RESULT_FOLDER_PATTERN = "tpwrs_*"
EXCLUDED_MODEL_SLUGS = {"mistral-small-2603"}

LOW_MAX = 40
MODERATE_MAX = 80
LEVEL_ORDER = ["Low", "Moderate", "High"]
LEVEL_BINS = [0, LOW_MAX, MODERATE_MAX, np.inf]

## 2. Load the per-model RA scores

One `ra_affinities.csv` per model per year folder. The publication year is taken from the folder
name (`tpwrs_2024_tpwrs_2024` -> `2024`).

In [ ]:
result_folders = sorted(folder for folder in BENCHMARK_ROOT.glob(RESULT_FOLDER_PATTERN) if folder.is_dir())

frames = []
for folder in result_folders:
    for score_path in sorted(folder.glob("*/ra_affinities.csv")):
        if score_path.parent.name in EXCLUDED_MODEL_SLUGS:
            continue
        frame = pd.read_csv(score_path, usecols=["Abstract_Index", "RA2025_ID", "RA_Question", "LLM_Affinity"])
        frame["Year"] = int(re.search(r"(\d{4})", folder.name).group(1))
        frame["Model_Slug"] = score_path.parent.name
        frames.append(frame)

ra_model_scores = pd.concat(frames, ignore_index=True)
ra_model_scores["LLM_Affinity"] = pd.to_numeric(ra_model_scores["LLM_Affinity"], errors="coerce")

YEARS = sorted(ra_model_scores["Year"].unique())
MODEL_SLUGS = sorted(ra_model_scores["Model_Slug"].unique())

print(f"{len(ra_model_scores):,} model-score rows")
print(f"years: {YEARS}")
print(f"models in the ensemble: {MODEL_SLUGS}")
ra_model_scores.head()

## 3. Coverage and model spread

Two things worth knowing before averaging: whether every model scored every year completely, and how
far apart the models' score levels are. The panel is deliberately heterogeneous, so the ensemble mean
sits between a conservative model (`gpt-5.4-mini`, `deepseek-v4-flash-cloud`) and a generous one
(`gemini-3.1-flash-lite`); a plain mean is the agreed reduction.

`Mean_Score` per year is also the cheapest check that any trend found later is not one model drifting:
the models are identical across years, so a rise should appear in every row, not just one.

In [ ]:
coverage = (
    ra_model_scores.groupby(["Model_Slug", "Year"])
    .agg(
        Rows=("LLM_Affinity", "size"),
        Unscored=("LLM_Affinity", lambda scores: scores.isna().sum()),
        Mean_Score=("LLM_Affinity", "mean"),
    )
    .round(1)
    .unstack("Year")
)
display(coverage)

print("abstracts per year:")
print(ra_model_scores.groupby("Year")["Abstract_Index"].nunique().to_string())

## 4. Ensemble mean and relevance level

One row per (year, abstract, question): the mean affinity across the models that returned a score,
then the Low / Moderate / High band.

In [ ]:
ra_pairs = (
    ra_model_scores.dropna(subset=["LLM_Affinity"])
    .groupby(["Year", "Abstract_Index", "RA2025_ID"], as_index=False)
    .agg(Affinity=("LLM_Affinity", "mean"), Models_Scored=("Model_Slug", "nunique"))
)
ra_pairs["Level"] = pd.cut(ra_pairs["Affinity"], bins=LEVEL_BINS, labels=LEVEL_ORDER, right=False)

print(f"{len(ra_pairs):,} abstract-question pairs")
print("\nmodels contributing to each pair:")
print(ra_pairs["Models_Scored"].value_counts().sort_index().to_string())
print("\noverall level mix (%):")
print(ra_pairs["Level"].value_counts(normalize=True).mul(100).round(2).to_string())

## 5. Attach the programme grouping

`ra_grouping_ra2025.csv` maps each RA2025 question to its programme. The programme order in the file
follows the question numbering, so it is kept as the categorical order used by every table and chart.

In [ ]:
ra_grouping = (
    pd.read_csv(GROUPING_PATH, usecols=["Grouping", "RA2025"])
    .rename(columns={"Grouping": "Programme", "RA2025": "RA2025_ID"})
)
PROGRAMME_ORDER = list(dict.fromkeys(ra_grouping["Programme"]))
question_text = ra_model_scores[["RA2025_ID", "RA_Question"]].drop_duplicates().set_index("RA2025_ID")["RA_Question"]

ra_pairs = ra_pairs.merge(ra_grouping, on="RA2025_ID", how="left")
ra_pairs["Programme"] = pd.Categorical(ra_pairs["Programme"], categories=PROGRAMME_ORDER, ordered=True)
abstracts_per_year = ra_pairs.groupby("Year")["Abstract_Index"].nunique()

print(f"questions without a programme: {ra_pairs['Programme'].isna().sum()}")
print(ra_grouping.groupby("Programme", sort=False).size().rename("questions").to_string())

## 6. Per-question summary

For each question and year: how many of that year's papers land in each level, and the same as a
percentage of the year's papers. `Share_Moderate_Or_High` is the single-number "how much of this
year's output speaks to this question" measure used by the charts below.

In [ ]:
question_year = (
    ra_pairs.pivot_table(index=["Programme", "RA2025_ID", "Year"], columns="Level",
                         values="Abstract_Index", aggfunc="count", fill_value=0, observed=True)
    .reindex(columns=LEVEL_ORDER, fill_value=0)
    .rename(columns=lambda level: f"N_{level}")
    .reset_index()
)
question_year["N_Abstracts"] = question_year["Year"].map(abstracts_per_year)
for level in LEVEL_ORDER:
    question_year[f"Share_{level}"] = question_year[f"N_{level}"] / question_year["N_Abstracts"] * 100
question_year["Share_Moderate_Or_High"] = question_year["Share_Moderate"] + question_year["Share_High"]

mean_affinity = ra_pairs.groupby(["RA2025_ID", "Year"], observed=True)["Affinity"].mean().round(2)
question_year = question_year.merge(mean_affinity.rename("Mean_Affinity"), on=["RA2025_ID", "Year"])
question_year["RA_Question"] = question_year["RA2025_ID"].map(question_text)
question_year = question_year.sort_values(["RA2025_ID", "Year"]).reset_index(drop=True)

question_year.head(8)

## 7. Per-programme relevance

A programme is a group of RA questions, so an abstract's relevance to a programme is the **highest
score it reaches on any question in that group**, banded into Low / Moderate / High on the same
thresholds. `Best_RA2025_ID` records which question drove the level.

These levels are **not exclusive**: every abstract carries one level for each of the six programmes,
so the same paper can be High for Stability and Low for DER. Shares therefore sum to 100% within a
programme-year, never across programmes — the last cell below shows how many programmes a typical
paper reaches at Moderate or above.

In [ ]:
programme_paper_best = (
    ra_pairs.loc[
        ra_pairs.groupby(["Year", "Abstract_Index", "Programme"], observed=True)["Affinity"].idxmax(),
        ["Year", "Abstract_Index", "Programme", "RA2025_ID", "Affinity"],
    ]
    .rename(columns={"RA2025_ID": "Best_RA2025_ID", "Affinity": "Best_Affinity"})
    .reset_index(drop=True)
)
programme_paper_best["Level"] = pd.cut(programme_paper_best["Best_Affinity"], bins=LEVEL_BINS, labels=LEVEL_ORDER, right=False)

programme_year = (
    programme_paper_best.pivot_table(index=["Programme", "Year"], columns="Level",
                                     values="Abstract_Index", aggfunc="count", fill_value=0, observed=True)
    .reindex(columns=LEVEL_ORDER, fill_value=0)
    .rename(columns=lambda level: f"N_{level}")
    .reset_index()
)
programme_year["N_Abstracts"] = programme_year["Year"].map(abstracts_per_year)
for level in LEVEL_ORDER:
    programme_year[f"Share_{level}"] = programme_year[f"N_{level}"] / programme_year["N_Abstracts"] * 100
programme_year["Share_Moderate_Or_High"] = programme_year["Share_Moderate"] + programme_year["Share_High"]

programme_year.round(1)

In [ ]:
programmes_reached = (
    programme_paper_best.assign(Above_Low=programme_paper_best["Level"] != "Low")
    .groupby(["Year", "Abstract_Index"], observed=True)["Above_Low"]
    .sum()
    .rename("Programmes_Above_Low")
)

print("How many of the 6 programmes each paper reaches at Moderate or High (% of papers):")
display(
    programmes_reached.groupby("Year")
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .unstack("Programmes_Above_Low")
    .fillna(0)
)

## 8. Headline: the relevance mix over time

Left panel: every abstract-question pair. Because each paper is scored against all 48 questions and
addresses only a few, ~88% of pairs are necessarily Low — this panel shows the drift of that baseline.

Right panel: each paper's best-matching question anywhere in the agenda, which is the readable version
of the same trend.

In [ ]:
paper_best = ra_pairs.groupby(["Year", "Abstract_Index"], as_index=False).agg(Best_Affinity=("Affinity", "max"))
paper_best["Best_Level"] = pd.cut(paper_best["Best_Affinity"], bins=LEVEL_BINS, labels=LEVEL_ORDER, right=False)
paper_level_counts = (
    paper_best.pivot_table(index="Year", columns="Best_Level", values="Abstract_Index",
                           aggfunc="count", fill_value=0, observed=True)
    .reindex(columns=LEVEL_ORDER, fill_value=0)
)
paper_best

In [ ]:
pair_level_counts = (
    ra_pairs.pivot_table(index="Year", columns="Level", values="Abstract_Index",
                         aggfunc="count", fill_value=0, observed=True)
    .reindex(columns=LEVEL_ORDER, fill_value=0)
)
pair_level_share = pair_level_counts.div(pair_level_counts.sum(axis=1), axis=0).mul(100)

paper_best = ra_pairs.groupby(["Year", "Abstract_Index"], as_index=False).agg(Best_Affinity=("Affinity", "max"))
paper_best["Best_Level"] = pd.cut(paper_best["Best_Affinity"], bins=LEVEL_BINS, labels=LEVEL_ORDER, right=False)
paper_level_counts = (
    paper_best.pivot_table(index="Year", columns="Best_Level", values="Abstract_Index",
                           aggfunc="count", fill_value=0, observed=True)
    .reindex(columns=LEVEL_ORDER, fill_value=0)
)
paper_level_share = paper_level_counts.div(paper_level_counts.sum(axis=1), axis=0).mul(100)

overall_year = pd.concat(
    {"Pair_Share": pair_level_share.round(2), "Paper_Share": paper_level_share.round(2)}, axis=1
)
overall_year

In [ ]:
year_labels = [f"{year}\n({abstracts_per_year[year]} papers)" for year in YEARS]
positions = np.arange(len(YEARS))
panels = [
    (pair_level_share, "Across all 48 questions"),
    (paper_level_share, "Each paper's best-matching question"),
]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (table, panel_title) in zip(axes, panels):
    left = np.zeros(len(table))
    for level in LEVEL_ORDER:
        values = table[level].to_numpy()
        bars = ax.barh(positions, values, left=left, label=level)
        ax.bar_label(bars, labels=[f"{value:.0f}" if value >= 7 else "" for value in values],
                     label_type="center")
        left = left + values
    ax.set_yticks(positions, year_labels)
    ax.invert_yaxis()
    ax.set_xlim(0, 100)
    ax.set_xlabel("Share of publication year (%)")
    ax.set_title(panel_title)

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncols=3)
fig.suptitle("Relevance of TPWRS output to the 2025 research agenda")
fig.tight_layout(rect=(0, 0.09, 1, 1))
plt.show()

## 9. Programme-level evolution

One panel per programme keeps the three levels legible; the line chart then isolates the High band so
the programmes can be compared directly.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(12, 7), sharex=True, sharey=True)
for ax, programme in zip(axes.ravel(), PROGRAMME_ORDER):
    block = programme_year[programme_year["Programme"] == programme].set_index("Year")
    bottom = np.zeros(len(block))
    for level in LEVEL_ORDER:
        values = block[f"Share_{level}"].to_numpy()
        ax.bar(block.index.astype(str), values, bottom=bottom, label=level)
        bottom = bottom + values
    ax.set_title(programme)
    ax.set_ylim(0, 100)
    ax.grid(axis="y")
    ax.set_axisbelow(True)
for ax in axes[:, 0]:
    ax.set_ylabel("Share of that year's papers (%)")
for ax in axes[-1, :]:
    ax.set_xlabel("Publication year")

handles, labels = axes[0, 0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower center", ncols=3)
fig.suptitle("Relevance level of the highest-scoring question in each programme")
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for programme in PROGRAMME_ORDER:
    block = programme_year[programme_year["Programme"] == programme].sort_values("Year")
    ax.plot(block["Year"], block["Share_High"], marker="o", label=programme)

ax.set_xticks(YEARS)
ax.set_ylim(0, None)
ax.set_xlabel("Publication year")
ax.set_ylabel("Papers with a High-relevance question (%)")
ax.set_title("Share of papers reaching High relevance in each programme")
ax.legend()
ax.grid(axis="y")
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

## 10. Per-question evolution

The heatmap is the full 48-question picture: each cell is the share of that year's papers scoring
Moderate or High on that question, with questions grouped by programme. The bar chart then ranks the
questions by how much that share moved between the first and last year available.

In [ ]:
question_heat = (
    question_year.pivot_table(index=["Programme", "RA2025_ID"], columns="Year",
                             values="Share_Moderate_Or_High", observed=True)
    .reindex(PROGRAMME_ORDER, level="Programme")
)
heat_values = question_heat.to_numpy()
heat_labels = [
    f"{programme} - Q{ra_id} {question_text[ra_id][:40]}..." for programme, ra_id in question_heat.index
]

fig, ax = plt.subplots(figsize=(10, 14))
image = ax.imshow(heat_values, aspect="auto")
ax.set_xticks(range(len(question_heat.columns)), [str(year) for year in question_heat.columns])
ax.set_yticks(range(len(question_heat)), heat_labels)

label_threshold = heat_values.max() / 2
for row in range(heat_values.shape[0]):
    for col in range(heat_values.shape[1]):
        ax.text(col, row, f"{heat_values[row, col]:.0f}", ha="center", va="center",
                color="white" if heat_values[row, col] < label_threshold else "black")

ax.set_xlabel("Publication year")
ax.set_title("Share of each year's papers at Moderate or High relevance, per question (%)")
fig.colorbar(image, ax=ax, fraction=0.03, label="% of that year's papers")
fig.tight_layout()
plt.show()

In [ ]:
FIRST_YEAR, LAST_YEAR = YEARS[0], YEARS[-1]

question_change = question_heat.reset_index()
question_change["Change_pp"] = question_change[LAST_YEAR] - question_change[FIRST_YEAR]
question_change = question_change.sort_values("Change_pp").reset_index(drop=True)

positions = np.arange(len(question_change))

fig, ax = plt.subplots(figsize=(9, 12))
ax.barh(positions, question_change["Change_pp"])
ax.set_yticks(
    positions,
    [f"{programme} - Q{ra_id}" for programme, ra_id in zip(question_change["Programme"], question_change["RA2025_ID"])],
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_xlabel(f"Change in Moderate-or-High share, {FIRST_YEAR} to {LAST_YEAR} (percentage points)")
ax.set_title(f"Which research questions TPWRS moved toward, {FIRST_YEAR} to {LAST_YEAR}")
ax.grid(axis="x")
ax.set_axisbelow(True)
fig.tight_layout()
plt.show()

## 11. Where the agenda is best and least served

Two shortlists from the latest year: the questions the journal engages with most, and the ones it
barely touches. The `Change_pp` column carries the movement since the first year, so a low-coverage
question that is climbing reads differently from one that is flat.

In [ ]:
latest = question_year[question_year["Year"] == LAST_YEAR].set_index("RA2025_ID")
shortlist = pd.DataFrame({
    "Programme": latest["Programme"],
    "Share_Moderate_Or_High": latest["Share_Moderate_Or_High"].round(1),
    "Share_High": latest["Share_High"].round(1),
    "Mean_Affinity": latest["Mean_Affinity"],
    "Change_pp": question_change.set_index("RA2025_ID")["Change_pp"].round(1),
    "RA_Question": latest["RA_Question"].str.slice(0, 90) + "...",
}).sort_values("Share_Moderate_Or_High", ascending=False)

print(f"Best-served questions in {LAST_YEAR}")
display(shortlist.head(10))
print(f"Least-served questions in {LAST_YEAR}")
display(shortlist.tail(10))